In [16]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
import pickle
import json
import matplotlib.pyplot as plt
from glob import glob
import time
import copy
from tqdm import tqdm
from transformers import BertLMHeadModel, BartTokenizer, BartForConditionalGeneration, BartConfig, BartForSequenceClassification, BertTokenizer, BertConfig, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification, PegasusForConditionalGeneration, PegasusTokenizer, T5Tokenizer, T5ForConditionalGeneration, BertGenerationEncoder, BertGenerationDecoder, EncoderDecoderConfig, EncoderDecoderModel
from data_singleword import otago_dataset
from model_decoding import BrainTranslator, BrainTranslatorNaive, T5Translator
from config import get_config

def show_require_grad_layers(model):
    print()
    print(' require_grad layers:')
    # sanity check
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(' ', name)


In [17]:
args = {
    "num_epoch_step1": 15,
    "num_epoch_step2": 25,
    "learning_rate_step1": 0.005,
    "learning_rate_step2": 0.005,
    "batch_size": 32,
    "model_name": "T5Translator",
    "task_name": "task1", 
    "train_input": "clean",
    "save_path": "/projects/open_vocab/saved_models",
    "subjects": "ALL",
    "eeg_type": "GD", 
    "eeg_bands": ["_t1", "_t2", "_a1", "_a2", "_b1", "_b2", "_g1", "_g2"],
    "cuda": 0,
    "skip_step_one": False,
    "load_step1_checkpoint": False,
    "use_random_init": True,
}

In [13]:
''' config param'''
dataset_setting = 'unique_sent'

num_epochs_step1 = args['num_epoch_step1']
num_epochs_step2 = args['num_epoch_step2']
step1_lr = args['learning_rate_step1']
step2_lr = args['learning_rate_step2']

batch_size = args['batch_size']

model_name = args['model_name']

task_name = args['task_name']
train_input = args['train_input']
print("train_input is:", train_input)   
save_path = args['save_path']
if not os.path.exists(save_path):
    os.makedirs(save_path)

skip_step_one = args['skip_step_one']
load_step1_checkpoint = args['load_step1_checkpoint']
use_random_init = args['use_random_init']
device_ids = [0] # device setting

if use_random_init and skip_step_one:
    step2_lr = 5*1e-4
    
print(f'[INFO]using model: {model_name}')

if skip_step_one:
    save_name = f'{task_name}_finetune_{model_name}_skipstep1_b{batch_size}_{num_epochs_step1}_{num_epochs_step2}_{step1_lr}_{step2_lr}_{dataset_setting}_{train_input}'
else:
    save_name = f'{task_name}_finetune_{model_name}_2steptraining_b{batch_size}_{num_epochs_step1}_{num_epochs_step2}_{step1_lr}_{step2_lr}_{dataset_setting}_{train_input}'

if use_random_init:
    save_name = 'randinit_' + save_name

save_path_best = os.path.join(save_path, 'best')
if not os.path.exists(save_path_best):
    os.makedirs(save_path_best)

output_checkpoint_name_best = os.path.join(save_path_best, f'{save_name}.pt')

save_path_last = os.path.join(save_path, 'last')
if not os.path.exists(save_path_last):
    os.makedirs(save_path_last)

output_checkpoint_name_last = os.path.join(save_path_last, f'{save_name}.pt')

# subject_choice = 'ALL
subject_choice = args['subjects']
print(f'![Debug]using {subject_choice}')
# eeg_type_choice = 'GD
eeg_type_choice = args['eeg_type']
print(f'[INFO]eeg type {eeg_type_choice}')
# bands_choice = ['_t1'] 
# bands_choice = ['_t1','_t2','_a1','_a2','_b1','_b2','_g1','_g2'] 
bands_choice = args['eeg_bands']
print(f'[INFO]using bands {bands_choice}')



''' set random seeds '''
seed_val = 312
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)


''' set up device '''
# use cuda
if torch.cuda.is_available():  
    # dev = "cuda:3" 
    dev = args['cuda'] 
else:  
    dev = "cpu"
# CUDA_VISIBLE_DEVICES=0,1,2,3  
device = torch.device(dev)
print(f'[INFO]using device {dev}')
print()

''' set up dataloader '''
whole_dataset_dicts = []
if 'task1' in task_name:
    dataset_path_task1 = '/eeg/processed_data/pickle/single_word_classification.pickle'
    with open(dataset_path_task1, 'rb') as handle:
        whole_dataset_dicts.append(pickle.load(handle))
if 'task2' in task_name:
    dataset_path_task2 = '/eeg/processed_data/pickle/single_word_classification_read.pickle'
    with open(dataset_path_task2, 'rb') as handle:
        whole_dataset_dicts.append(pickle.load(handle))

print()

"""save config"""
cfg_dir = './config/decoding/'

if not os.path.exists(cfg_dir):
    os.makedirs(cfg_dir)

with open(os.path.join(cfg_dir,f'{save_name}.json'), 'w') as out_config:
    json.dump(args, out_config, indent = 4)

if model_name in ['BrainTranslator','BrainTranslatorNaive']:
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

elif model_name == 'PegasusTranslator':
    tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-xsum')

elif model_name == 'T5Translator':
    tokenizer = T5Tokenizer.from_pretrained("t5-large")
    #tokenizer.set_prefix_tokens(language='english')

# train dataset
train_set = otago_dataset(whole_dataset_dicts, 'train', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# dev dataset
dev_set = otago_dataset(whole_dataset_dicts, 'dev', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# test dataset
# test_set = otago_dataset(whole_dataset_dicts, 'test', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting)


dataset_sizes = {'train': len(train_set), 'dev': len(dev_set)}
print('[INFO]train_set size: ', len(train_set))
print('[INFO]dev_set size: ', len(dev_set))
# print('[INFO]test_set size: ', len(test_set))

# train dataloader
train_dataloader = DataLoader(train_set, batch_size = batch_size, shuffle=True, num_workers=4)
# dev dataloader
val_dataloader = DataLoader(dev_set, batch_size = batch_size, shuffle=False, num_workers=4)
# dataloaders
dataloaders = {'train':train_dataloader, 'dev':val_dataloader}

''' set up model '''
if model_name == 'BrainTranslator':
    if use_random_init:
        config = BartConfig.from_pretrained('facebook/bart-large')
        pretrained = BartForConditionalGeneration(config)
    else:
        pretrained = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

    model = BrainTranslator(pretrained, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

elif model_name == 'BrainTranslatorNaive':
    pretrained = BartForConditionalGeneration.from_pretrained('facebook/bart-large')
    model = BrainTranslatorNaive(pretrained, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

elif model_name == 'PegasusTranslator':
    pretrained = PegasusForConditionalGeneration.from_pretrained('google/pegasus-xsum')
    model = BrainTranslator(pretrained, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

elif model_name == 'T5Translator':
    pretrained = T5ForConditionalGeneration.from_pretrained("t5-large")
    model = T5Translator(pretrained, in_feature = 64*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

model.to(device)
model = torch.nn.DataParallel(model, device_ids=device_ids)

''' training loop '''

######################################################
'''step one trainig: freeze most of BART params'''
######################################################

# closely follow BART paper
if model_name in ['BrainTranslator','BrainTranslatorNaive', 'PegasusTranslator', 'T5Translator']:
    for name, param in model.named_parameters():
        if param.requires_grad and 'pretrained' in name:
            if ('shared' in name) or ('embed_positions' in name) or ('encoder.layers.0' in name):
                continue
            else:
                param.requires_grad = False

elif model_name == 'BertGeneration':
    for name, param in model.named_parameters():
        if param.requires_grad and 'pretrained' in name:
            if ('embeddings' in name) or ('encoder.layer.0' in name):
                continue
            else:
                param.requires_grad = False

train_input is: clean
[INFO]using model: T5Translator
![Debug]using ALL
[INFO]eeg type GD
[INFO]using bands ['_t1', '_t2', '_a1', '_a2', '_b1', '_b2', '_g1', '_g2']
[INFO]using device 0


[INFO] Loading 1 dataset(s)
[INFO] Input tensor size: torch.Size([2, 512])
[INFO] Total samples loaded: 3520

[INFO] Loading 1 dataset(s)
[INFO] Input tensor size: torch.Size([2, 512])
[INFO] Total samples loaded: 440

[INFO]train_set size:  3520
[INFO]dev_set size:  440


In [8]:
verbose = False

def train_model(dataloaders, device, model, criterion, optimizer, scheduler, num_epochs=25, checkpoint_path_best = './checkpoints/decoding/best/temp_decoding.pt', checkpoint_path_last = './checkpoints/decoding/last/temp_decoding.pt'):
    # modified from: https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
    since = time.time()
      
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = 100000000000

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'dev']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0

            # Iterate over data.
            for input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels in tqdm(dataloaders[phase]):

                if verbose:
                    print("input_embeddings", input_embeddings.shape, input_embeddings)
                    print("seq_len", seq_len.shape, seq_len)
                    print("input_masks", input_masks.shape, input_masks)
                    print("input_mask_invert", input_mask_invert.shape, input_mask_invert)
                    print("target_ids", target_ids.shape, target_ids)
                    print("target_mask", target_mask.shape, target_mask)
                    print("sentiment_labels", sentiment_labels.shape, sentiment_labels)
                    # print("target_ids_first_only", target_ids_first_only.shape, target_ids_first_only)
                
                # load in batch
                input_embeddings_batch = input_embeddings.to(device).float()
                input_masks_batch = input_masks.to(device)
                input_mask_invert_batch = input_mask_invert.to(device)
                target_ids_batch = target_ids.to(device)
                """replace padding ids in target_ids with -100"""
                target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100

                # print("Target IDs before masking:", target_ids_batch[0, :5].tolist())
                # valid_ids = target_ids_batch[0, :5][target_ids_batch[0, :5] != -100]
                # print("Target tokens:", tokenizer.convert_ids_to_tokens(valid_ids.tolist(), skip_special_tokens=False))
                # print("After masking:", target_ids_first_only[0, :5].tolist())

                # Mask all tokens except the first non-pad token
                target_ids_first_only = target_ids_batch.clone()
                target_ids_first_only[:, 1:] = -100

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
    	        # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    seq2seqLMoutput = model(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_first_only)

                    """calculate loss"""
                    logits = seq2seqLMoutput.logits # 8*48*50265
                    # logits = logits.permute(0,2,1) # 8*50265*48

                    # loss = criterion(logits, target_ids_batch_label) # calculate cross entropy loss only on encoded target parts
                    # NOTE: my criterion not used
                    loss = seq2seqLMoutput.loss # use the BART language modeling loss

                    # print("loss:", loss)

                    """check prediction, instance 0 of each batch"""
                    # print('target size:', target_ids_batch.size(), ',original logits size:', logits.size(), ',target_mask size', target_mask_batch.size())
                    # print('target size:', target_ids_batch.size(), ',original logits size:', logits.size())
                    # print("original logits:", logits)
                    # print("original logits size:", logits.size())
                    # logits = logits.permute(0,2,1)
                    # print("after permute logits:", logits)
                    # print("after permute logits size:", logits.size())
                    for idx in [0]:
                        # print(f'-- instance {idx} -')
                        # print('permuted logits size:', logits.size())
                        probs = logits[idx].softmax(dim = 1)
                        # print("logits[0]:", logits[0])
                        # print('probs size:', probs.size())
                        # print('probs:', probs)
                        
                        values, predictions = probs.topk(1)
                        # print('predictions before squeeze:',predictions.size())
                        predictions = torch.squeeze(predictions)
                        # print('predictions:',predictions)
                        # print("decoded predictions:", tokenizer.decode([predictions[0]], skip_special_tokens=False))
                        # print('target mask:', target_mask_batch[idx])
                        # print('[DEBUG]target tokens:',tokenizer.decode(target_ids_batch_copy[idx]))
                        # print('[DEBUG]target tokens:',tokenizer.decode(target_ids_batch[idx]))
                        valid_target_ids = target_ids_batch[idx][target_ids_batch[idx] != -100]

                        if verbose:
                            print('[DEBUG]target tokens:', tokenizer.decode(valid_target_ids, skip_special_tokens=True))
                        
                         # FIRST TOKEN ONLY
                        first_token_logits = logits[idx, 0]        # (V,)
                        # print("first_token_logits", first_token_logits.size(), first_token_logits)
                        first_token_id = first_token_logits.argmax(dim=-1)
                        # print("first_token_id", first_token_id)

                        if verbose:
                            print('[DEBUG]predicted tokens:',tokenizer.decode([first_token_id], skip_special_tokens=False), " id:", first_token_id)

                        target_id = valid_target_ids[0].item()

                        # print("target token == predicted token:", target_id == first_token_id.item(), "target:", target_id, "predict:", first_token_id.item())

                        probs = first_token_logits.softmax(dim=-1)   # shape: [vocab_size]
                        target_prob = probs[target_id].item()
                        top5_probs, top5_ids = probs.topk(5)
                        # print("Target token prob:", target_prob)
                        # print("Top 5 predictions:", top5_ids.tolist(), top5_probs.tolist())

                        # Sort descending to get ranks
                        sorted_probs, sorted_indices = probs.sort(descending=True)
                        
                        # Find rank
                        rank_tensor = (sorted_indices == target_id).nonzero(as_tuple=False)
                        
                        if rank_tensor.numel() == 0:
                            rank = -1  # target not found
                        else:
                            # rank_tensor could be 2D, take the first row and first column
                            rank = rank_tensor[0, 0].item() + 1  # 1-based rank
                        
                        # print(f"Target token id: {target_id}, rank in predictions: {rank}")


                
                        # valid = target_ids_first_only[idx][target_ids_first_only[idx] != -100]
                        # print("LOSS TOKENS:", tokenizer.convert_ids_to_tokens(valid))
                
                    # return
                
                    # backward + optimize only if in training phase
                    if phase == 'train':
                        # with torch.autograd.detect_anomaly():
                        loss.sum().backward()
                        optimizer.step()

                # statistics
                running_loss += loss.sum().item() * input_embeddings_batch.size()[0] # batch loss
                # print('[DEBUG]loss:',loss.item())
                # print('#################################')
                

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]

            print('{} Loss: {:.4f}'.format(phase, epoch_loss))

            # deep copy the model
            if phase == 'dev' and epoch_loss < best_loss:
                best_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                '''save checkpoint'''
                # torch.save(model.state_dict(), checkpoint_path_best)
                # print(f'update best on dev checkpoint: {checkpoint_path_best}')
                
                # with torch.set_grad_enabled(False):
                #     traced_model_1 = torch.jit.trace(model, (torch.rand(1, 56, 840).to(device), torch.randint(1, 56).to(device), torch.rand(1, 56).to(device), torch.rand(1, 56).to(device)))
                #     traced_model_32 = torch.jit.trace(model, (torch.rand(32, 56, 840).to(device), torch.randint(32, 56).to(device), torch.rand(32, 56).to(device), torch.rand(32, 56).to(device)))
                # torch.jit.save(traced_model_1, checkpoint_path_best[:-3]+'_1_jit.pt')
                # torch.jit.save(traced_model_32, checkpoint_path_best[:-3]+'_32_jit.pt')
        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
    print('Best val loss: {:4f}'.format(best_loss))
    torch.save(model.state_dict(), checkpoint_path_last)
    print(f'update last checkpoint: {checkpoint_path_last}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [14]:
if skip_step_one:
    if load_step1_checkpoint:
        stepone_checkpoint = 'path_to_step_1_checkpoint.pt'
        print(f'skip step one, load checkpoint: {stepone_checkpoint}')
        model.load_state_dict(torch.load(stepone_checkpoint))
    else:
        print('skip step one, start from scratch at step two')
else:

    ''' set up optimizer and scheduler'''
    optimizer_step1 = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=step1_lr, momentum=0.9)

    exp_lr_scheduler_step1 = lr_scheduler.StepLR(optimizer_step1, step_size=20, gamma=0.1)

    ''' set up loss function '''
    criterion = nn.CrossEntropyLoss()

    print('=== start Step1 training ... ===')
    # print training layers
    # show_require_grad_layers(model)
    # return best loss model from step1 training
    model = train_model(dataloaders, device, model, criterion, optimizer_step1, exp_lr_scheduler_step1, num_epochs=num_epochs_step1, checkpoint_path_best = output_checkpoint_name_best, checkpoint_path_last = output_checkpoint_name_last)

######################################################
'''step two trainig: update whole model for a few iterations'''
######################################################
for name, param in model.named_parameters():
    param.requires_grad = True

''' set up optimizer and scheduler'''
optimizer_step2 = optim.SGD(model.parameters(), lr=step2_lr, momentum=0.9)

exp_lr_scheduler_step2 = lr_scheduler.StepLR(optimizer_step2, step_size=30, gamma=0.1)

''' set up loss function '''
criterion = nn.CrossEntropyLoss()

print()
print('=== start Step2 training ... ===')
# print training layers
# show_require_grad_layers(model)

'''main loop'''
trained_model = train_model(dataloaders, device, model, criterion, optimizer_step2, exp_lr_scheduler_step2, num_epochs=num_epochs_step2, checkpoint_path_best = output_checkpoint_name_best, checkpoint_path_last = output_checkpoint_name_last)


=== start Step1 training ... ===
Epoch 0/14
----------


  0%|          | 0/110 [00:00<?, ?it/s]

input_embeddings_batch shape torch.Size([32, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([32, 2, 512])


  1%|          | 1/110 [00:01<01:57,  1.08s/it]

input_embeddings_batch shape torch.Size([32, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([32, 2, 512])


...

input_embeddings_batch shape torch.Size([32, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([32, 2, 512])


 25%|██▍       | 27/110 [00:06<00:19,  4.16it/s]

input_embeddings_batch shape torch.Size([32, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([32, 2, 512])


 25%|██▍       | 27/110 [00:06<00:21,  3.89it/s]


KeyboardInterrupt: 